# 📱 Mobile Money Data Extractor — v2
**CSC 3221 — Introduction to Data Science | ICT University**

Fully automated pipeline that processes **all** SMS export files in `data/` and produces clean, anonymized transaction datasets in `output/`.

### What it does
1. Auto-discovers every `.csv` / `.xlsx` file in `data/`
2. Detects the operator (OrangeMoney / MobileMoney) and file owner automatically
3. Filters to balance-changing messages only
4. Extracts: transaction type, direction, amount, currency, new balance
5. Anonymizes: owner → `[USER_N]` / `[USER_N_phone]` · third-parties → `[CONTACT_XXXX]` / `[PHONE_XXXX]`
6. Exports a styled `.xlsx` + `.csv` per file, and a shared `owner_map.json`

**Just drop your files in `data/` and run all cells. No configuration needed.**

> Supported formats: CSV & Excel · Supported languages: French & English


## ⚙️ Step 1 — Install & imports

In [1]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'openpyxl', 'chardet', '--quiet'], check=False)

import re
import json
import hashlib
import unicodedata
from collections import Counter
from pathlib import Path

import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

print('Ready!')


Ready!


## 📂 Step 2 — Folder setup

In [2]:
NOTEBOOK_DIR = Path().resolve()
DATA_DIR     = NOTEBOOK_DIR / 'data'
OUTPUT_DIR   = NOTEBOOK_DIR / 'output'
MAP_FILE     = OUTPUT_DIR   / 'owner_map.json'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SUPPORTED = {'.csv', '.xlsx', '.xls'}
files = sorted(f for f in DATA_DIR.iterdir()
               if f.is_file() and f.suffix.lower() in SUPPORTED)

if not files:
    print(f'No files found in {DATA_DIR}. Add your SMS export files and re-run.')
else:
    print(f'DATA_DIR   : {DATA_DIR}')
    print(f'OUTPUT_DIR : {OUTPUT_DIR}')
    print(f'Files found: {len(files)}')
    for f in files:
        print(f'  - {f.name}')


DATA_DIR   : C:\Users\joelf\Documents\GitHub\mobile_money_analysis\data
OUTPUT_DIR : C:\Users\joelf\Documents\GitHub\mobile_money_analysis\output
Files found: 23
  - Messages avec MobileMoney 2026-03-20 21-32-36 - Madeleine Tchayo.csv
  - Messages avec MobileMoney 2026-03-22 12_27_25 - Jacques Fah.csv
  - Messages avec OrangeMoney 2026-03-20 21-31-49 - Madeleine Tchayo.csv
  - Messages avec OrangeMoney 2026-03-20 22_05_33 - Pascal Esaïe BOUGONG A ABEGA.csv
  - Messages avec OrangeMoney 2026-03-21 11_16_17 - DAVIS JERRY.csv
  - Messages avec OrangeMoney 2026-03-22 12_28_16 - Jacques Fah.csv
  - Messages avec OrangeMoney 2026-03-23 014528 - Manuel Karim.csv
  - Messages avec OrangeMoney Brown 2026-03-23 083311 - Brown Takou.csv
  - Messages with MobileMoney 2026-03-17 214651.xlsx
  - Messages with MobileMoney 2026-03-20 21_13_48 - Dejon Fah Joël Xavier.csv
  - Messages with MobileMoney 2026-03-23 073413.csv
  - Messages with MobileMoney 2026-03-23 210438.xlsx
  - Messages with MobileMone

## 📖 Step 3 — File loading & column normalisation

In [3]:
# FR/EN column name aliases -> canonical English
COL_ALIASES = {
    'date':      ['date'],
    'time':      ['heure', 'time'],
    'direction': ['direction'],
    'contact':   ['contact'],
    'phone':     ['telephone', 'phone'],
    'content':   ['contenu', 'content'],
    'type':      ['type'],
}

def normalize_columns(df):
    rename = {}
    for canonical, aliases in COL_ALIASES.items():
        for col in df.columns:
            if str(col).strip().lower().replace('\u00e9','e') in aliases:
                rename[col] = canonical
                break
    return df.rename(columns=rename)

ENCODINGS = ['utf-8-sig', 'utf-8', 'latin-1', 'cp1252']

def detect_encoding(path):
    try:
        import chardet
        result = chardet.detect(path.read_bytes()[:8192])
        if result.get('encoding') and result['confidence'] > 0.7:
            return result['encoding']
    except ImportError:
        pass
    for enc in ENCODINGS:
        try:
            with open(path, encoding=enc) as f: f.read(4096)
            return enc
        except (UnicodeDecodeError, LookupError):
            continue
    return 'latin-1'

def read_csv_robust(path, encoding, skiprows):
    with open(path, encoding=encoding, errors='replace') as f:
        all_lines = f.readlines()
    data_lines = all_lines[skiprows:]
    if not data_lines:
        raise pd.errors.EmptyDataError('No data after header skip')
    header_raw = data_lines[0].rstrip('\r\n')
    sep    = '\t' if '\t' in header_raw else ','
    header = [h.strip().strip('"') for h in header_raw.split(sep)]
    ncols  = len(header)
    rows = []
    for line in data_lines[1:]:
        line = line.rstrip('\r\n')
        if not line.strip(): continue
        parts = line.split(sep, ncols - 2) if sep == ',' else line.split('\t')
        if sep == ',' and len(parts) == ncols - 1:
            lc = parts[-1].rfind(',')
            if lc != -1:
                parts = parts[:-1] + [parts[-1][:lc], parts[-1][lc + 1:]]
        parts = (parts + [''] * ncols)[:ncols]
        rows.append([p.strip().strip('"') for p in parts])
    return pd.DataFrame(rows, columns=header)

EXPECTED_COLS = {'content', 'contenu', 'date'}
CSV_SKIP = 3

def load_file(path):
    ext = path.suffix.lower()
    if ext == '.csv':
        encoding = detect_encoding(path)
        for skip in (CSV_SKIP, 0):
            try:
                df = read_csv_robust(path, encoding, skip)
            except pd.errors.EmptyDataError:
                continue
            df = normalize_columns(df)
            if EXPECTED_COLS.intersection(df.columns): break
        else:
            raise ValueError(f'Could not find expected columns in {path.name!r}')
    elif ext in ('.xlsx', '.xls'):
        for skip in (CSV_SKIP, 0):
            df = pd.read_excel(path, skiprows=skip, header=0, dtype=str)
            df = normalize_columns(df)
            if EXPECTED_COLS.intersection(df.columns): break
        else:
            raise ValueError(f'Could not find expected columns in {path.name!r}')
    else:
        raise ValueError(f'Unsupported file type: {ext}')
    df['content'] = (
        df['content'].astype(str)
        .str.replace('_x000d_', ' ', regex=False).str.strip()
    )
    return df

print('File loading utilities ready!')


File loading utilities ready!


## 🏦 Step 4 — Operator & owner detection

In [4]:
def detect_operator(df):
    for col in ['contact', 'phone']:
        if col in df.columns:
            sample = df[col].dropna().astype(str).str.lower()
            if sample.str.contains('orangemoney', na=False).any(): return 'OrangeMoney'
            if sample.str.contains('mobilemoney', na=False).any(): return 'MobileMoney'
    return 'Unknown'

PHONE_RE_DETECT = re.compile(r'\b(237)?(6\d{8}|2\d{8})\b')
MTN_OWNER_RE = re.compile(
    r'(\b[A-Z\u00c0-\u00dd][A-Z\u00c0-\u00dda-z\u00e0-\u00fd]+'
    r'(?:\s+[A-Z\u00c0-\u00dda-z\u00e0-\u00fd&][A-Z\u00c0-\u00dda-z\u00e0-\u00fd&]+)*)'
    r'\s+\(237(\d{9})\s*\)'
)
OM_OWNER_RE = re.compile(
    r'\b((?:237)?(?:6\d{8}|2\d{8}))\s+'
    r'((?:[A-Z\u00c0-\u00dda-z\u00e0-\u00fd&][A-Z\u00c0-\u00dda-z\u00e0-\u00fd&]+)'
    r'(?:\s+(?!to\b|vers\b|avec\b|reussi\b|Informations\b)'
    r'[A-Z\u00c0-\u00dda-z\u00e0-\u00fd&][A-Z\u00c0-\u00dda-z\u00e0-\u00fd&]+){0,4})'
    r'(?=\s+(?:to|vers|avec|reussi|Informations)|\s*[.\n,])'
)
NAME_STOPWORDS = {'from','to','of','by','de','du','par','vers','avec',
                  'le','la','les','un','une','your','xaf','fcfa'}

def strip_accents(s):
    return unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode()

def norm_name(s):
    return re.sub(r'\s+', ' ', strip_accents(str(s)).strip().lower())

def bare_phone(raw):
    s = str(raw).strip().replace(' ', '')
    return s[3:] if s.startswith('237') and len(s) == 12 else s

def clean_name(raw):
    words = raw.split()
    while words and words[0].lower() in NAME_STOPWORDS:
        words = words[1:]
    return ' '.join(words)

def detect_owner_phone(df):
    c = Counter()
    for text in df['content'].dropna():
        for m in PHONE_RE_DETECT.finditer(str(text)):
            c[m.group(2)] += 1
    return c.most_common(1)[0][0] if c else None

def detect_owner_name(df, owner_phone):
    if not owner_phone: return None
    for text in df['content'].dropna():
        text = str(text)
        for m in MTN_OWNER_RE.finditer(text):
            if bare_phone('237' + m.group(2)) == owner_phone:
                n = clean_name(m.group(1))
                if n: return n
        for m in OM_OWNER_RE.finditer(text):
            if bare_phone(m.group(1)) == owner_phone:
                n = clean_name(m.group(2))
                if n: return n
    return None

def owner_from_filename(path):
    stem = path.stem
    for sep in (' - ', '_-_'):
        if sep in stem:
            return stem.rsplit(sep, 1)[1].replace('_', ' ').strip()
    return None

print('Operator & owner detection ready!')


Operator & owner detection ready!


## 🔍 Step 5 — Filter balance-changing messages

In [5]:
BALANCE_KEYWORDS = [
    'nouveau solde', 'nouveau solde est',
    'new balance',   'your new balance',
]
BALANCE_RE = re.compile(
    r'(?:' + '|'.join(re.escape(k) for k in BALANCE_KEYWORDS) + r')',
    re.IGNORECASE
)

def filter_balance_messages(df):
    mask = df['content'].apply(lambda x: bool(BALANCE_RE.search(str(x))))
    return df[mask].copy().reset_index(drop=True)

print('Balance filter ready!')


Balance filter ready!


## 🧠 Step 6 — Extract amount, currency, new balance

In [6]:
AMOUNT_RE = re.compile(
    r'(?:montant[^:]*:\s*|(?<!solde\sest\s)de\s+|amount\s+|of\s+)'
    r'(\d[\d\s,\.]*?)\s*(FCFA|XAF)',
    re.IGNORECASE
)
AMOUNT_FALLBACK_RE = re.compile(r'(\d[\d\s]*?)\s*(FCFA|XAF)', re.IGNORECASE)

BALANCE_VAL_RE = re.compile(
    r'(?:'
        r'votre\s+nouveau\s+solde\s+est\s+de\s+|your\s+new\s+balance\s+is\s+|'
        r'nouveau\s+solde\s+est\s+de\s*:?|new\s+balance\s+is\s*:?|'
        r'nouveau\s+solde\s+est\s*:?|new\s+balance\s+is\s*:?|'
        r'nouveau\s+solde\s*:?|new\s+balance\s*:?|'
        r'solde\s*:?|balance\s*:?'
    r')\s*(\d[\d\s,\.]*?)\s*(FCFA|XAF)',
    re.IGNORECASE
)

def clean_amount(raw):
    if raw is None: return None
    try:    return float(re.sub(r'[\s,]', '', str(raw)))
    except: return None

def extract_amount(text):
    m = AMOUNT_RE.search(text) or AMOUNT_FALLBACK_RE.search(text)
    return (clean_amount(m.group(1)), m.group(2).upper()) if m else (None, None)

def extract_new_balance(text):
    m = BALANCE_VAL_RE.search(text)
    return (clean_amount(m.group(1)), m.group(2).upper()) if m else (None, None)

print('Amount & balance extractors ready!')


Amount & balance extractors ready!


## 🏷️ Step 7 — Transaction classification

In [7]:
TX_RULES = [
    ('retrait', 'OUT', [
        r"retrait\s+d'argent",
        r'retrait\s+de\s+\d',
        r'vous\s+avez\s+effectue\s+avec\s+succes\s+le\s+retrait',
        r'withdrawal\s+successful',
        r'cash\s+out',
        r'you\s+have\s+successfully\s+withdrawn',
        r'withdrawn\s+from\s+your\s+mobile\s+money',
        r'you\s+have\s+withdrawn\s+\d+\s*(xaf|fcfa)',
        r'have\s+via\s+agent.*withdrawn',
    ]),
    ('depot', 'IN', [
        r'depot\s+effectue\s+par',
        r'deposit\s+made\s+by',
        r'deposit\s+to\s+your',
        r'you\s+have\s+received\s+a\s+deposit\s+of',
    ]),
    ('transfert', 'IN', [
        r'vous\s+avez\s+re[c\u00e7]u\s+\d',
        r'vous\s+avez\s+recu\s+avec\s+succes',               # OM: 'Vous avez recu avec succes N FCFA de'
        r'you\s+have\s+received\s+\d',
        r'has\s+been\s+added\s+to\s+your',
        r'adjustment\s+has\s+been\s+made',
        r'you\s+just\s+received',
        r'you\s+have\s+received.*in\s+your\s+mobile\s+money',
        r'vous\s+avez\s+re[c\u00e7]u.*xaf',
    ]),
    ('transfert', 'OUT', [
        r'transfert\s+de\s+\d+\s*(fcfa|xaf)',
        r'transfer\s+of\s+\d',
        r'transfert.*effectue.*succes.*\d',
        r'transfert.*vers\s+\d',
        r'successful\s+transfer\s+.*\s+xaf\s+to',
        r'you\s+have\s+transferred\s+\d',
    ]),
    ('paiement', 'OUT', [
        r'paiement\s+de\s+votre\s+facture',
        r'votre\s+paiement\s+de\s+\d',
        r'your\s+payment\s+of\s+\d',
        r'paiement.*r[\u00e9e]ussi',
        r'payment.*successful',
        r'paiement\s+total',
        r'vous\s+venez\s+d.effectuer\s+un\s+pa[yi]e?ment',   # covers 'payement' variant
        r'has\s+been\s+completed',
    ]),
    ('rechargement', 'OUT', [
        r'rechargement\s+reussi',
        r'top.?up\s+successful',
        r'recharge\s+successful',
    ]),
    ('airtime', 'OUT', [
        r'achete\s+avec\s+succes.*airtime',
        r'airtime.*transaction',
        r'paiement.*airtime',
        r'payment.*airtime',
        r'you\s+have\s+received.*airtime\s+from',
        r'recu.*xaf\s+airtime',
        r'received.*xaf\s+airtime',
    ]),
    ('transaction', 'OUT', [
        r'une\s+transaction\s+de\s+\d',
        r'a\s+transaction\s+of\s+\d',
        r'transaction.*effectuee\s+par',
        r'transaction.*made\s+by',
        r'from\s+your\s+mobile\s+money\s+account\s+by',
        r'from\s+your\s+mobile\s+money\s+account.*completed',
    ]),
]

TX_RULES_COMPILED = [
    (t, d, [re.compile(p, re.IGNORECASE) for p in pats])
    for t, d, pats in TX_RULES
]

def classify_transaction(text):
    for tx_type, direction, compiled in TX_RULES_COMPILED:
        for pat in compiled:
            if pat.search(text):
                return tx_type, direction
    return 'autre', 'unknown'

print('Transaction classification rules ready!')


Transaction classification rules ready!


## 🔒 Step 8 — Anonymization engine

In [8]:
def short_hash(value, length=4):
    return hashlib.md5(str(value).encode()).hexdigest()[:length].upper()

PHONE_RE = re.compile(r'\b\d{9,12}\b')

NAME_AFTER_PHONE_RE = re.compile(
    r'\b(\d{9,12})\s*[-\u2013:]?\s*'
    r'([A-Z\u00c0\u00c2\u00c9\u00c8\u00ca\u00cb\u00ce\u00cf\u00d4\u00d9\u00db\u00dc\u00c7]{2,}'
    r'(?:\s+[A-Z\u00c0\u00c2\u00c9\u00c8\u00ca\u00cb\u00ce\u00cf\u00d4\u00d9\u00db\u00dc\u00c7]{2,}){0,3})\b'
)
NAME_BEFORE_PHONE_RE = re.compile(
    r'\b([A-Z\u00c0\u00c2\u00c9\u00c8\u00ca\u00cb\u00ce\u00cf\u00d4\u00d9\u00db\u00dc\u00c7]{2,}'
    r'(?:\s+[A-Z\u00c0\u00c2\u00c9\u00c8\u00ca\u00cb\u00ce\u00cf\u00d4\u00d9\u00db\u00dc\u00c7]{2,}){0,3})'
    r'\s*\((\d{9,12}\s*)\)'
)
WITHDRAWAL_NAME_RE = re.compile(
    r'\b(?:withdrawn|withdraw|retrait)\b.*?\b(?:chez|at)\s*[:\-\u2013]?\s*'
    r'([A-Z\u00c0\u00c2\u00c9\u00c8\u00ca\u00cb\u00ce\u00cf\u00d4\u00d9\u00db\u00dc\u00c7]{2,}'
    r'(?:\s+[A-Z\u00c0\u00c2\u00c9\u00c8\u00ca\u00cb\u00ce\u00cf\u00d4\u00d9\u00db\u00dc\u00c7]{2,}){0,4})\b',
    re.IGNORECASE
)
SYSTEM_TOKENS = {
    'FCFA','XAF','SMS','ID','MTN','ORANGE','MOBILEMONEY','OM','MOMO','OTP','PIN',
    'ENEO','CAMWATER','CANAL','DSTV','YELLO','MTNC','SWITCHN',
}

def _phone_variants(phone):
    if not phone: return []
    phone = str(phone).strip()
    variants = {phone}
    if re.match(r'^6\d{8}$', phone): variants.add('237' + phone)
    if re.match(r'^237\d{9}$', phone): variants.add(phone[3:])
    return list(variants)

def anonymize_message(text, user_name, user_phone, user_id):
    result = str(text)

    # 1. User name (case-insensitive, exact match)
    if user_name and user_name.strip():
        result = re.sub(re.escape(user_name.strip()),
                        f'[{user_id}]', result, flags=re.IGNORECASE)

    # 2. Withdrawal agent names (chez / at)
    def repl_withdrawal(m):
        name = m.group(1).strip()
        return m.group(0).replace(name, f'[CONTACT_{short_hash(name)}]')
    result = WITHDRAWAL_NAME_RE.sub(repl_withdrawal, result)

    # 3. Names AFTER phone (OM pattern) — must run before phone masking
    def repl_after(m):
        return f'{m.group(1)} [CONTACT_{short_hash(m.group(2).strip())}]'
    result = NAME_AFTER_PHONE_RE.sub(repl_after, result)

    # 4. Names BEFORE phone (MTN pattern 'NAME (237XXXXXXXXX)')
    def repl_before(m):
        return f'[CONTACT_{short_hash(m.group(1).strip())}] ({m.group(2).strip()})'
    result = NAME_BEFORE_PHONE_RE.sub(repl_before, result)

    # 5. User phone (local + international variants)
    for variant in _phone_variants(user_phone):
        result = result.replace(variant, f'[{user_id}_phone]')

    # 6. Remaining phone-like numbers
    result = PHONE_RE.sub(lambda m: f'[PHONE_{m.group(0)[-4:]}]', result)

    # 7. Remaining ALL-CAPS multi-word names (fallback)
    def repl_caps(m):
        name  = m.group(0).strip()
        words = name.split()
        if len(words) < 2 or any(w in SYSTEM_TOKENS for w in words):
            return name
        return f'[CONTACT_{short_hash(name)}]'
    result = re.sub(
        r'\b([A-Z\u00c0-\u00d6\u00d8-\u00dd]{2,}'
        r'(?:\s+[A-Z\u00c0-\u00d6\u00d8-\u00dd]{2,}){1,3})\b',
        repl_caps, result
    )
    return result

print('Anonymization engine ready!')


Anonymization engine ready!


## 💾 Step 9 — Styled Excel export helper

In [9]:
HEADER_COLOR = '1F3864'
IN_COLOR     = 'C6EFCE'
OUT_COLOR    = 'FFC7CE'
ALT_COLOR    = 'F2F9FF'
COL_WIDTHS   = {
    'UserId':14,'Date':12,'Time':10,'Operator':14,
    'Transaction_type':16,'Direction':10,
    'Amount':11,'Currency':9,'New_balance':13,'Anonymized_Content':70,
}

def thin_border(color='BBBBBB'):
    s = Side(style='thin', color=color)
    return Border(left=s, right=s, top=s, bottom=s)

def export_xlsx(df, path, user_id, operator):
    wb = Workbook()
    ws = wb.active
    ws.title = user_id[:31]
    ws.sheet_view.showGridLines = False
    n = len(df.columns)
    ws.merge_cells(f'A1:{get_column_letter(n)}1')
    ws['A1'] = f'Anonymized Mobile Money Transactions - {user_id} ({operator})'
    ws['A1'].font      = Font(name='Arial', bold=True, size=13, color='FFFFFF')
    ws['A1'].fill      = PatternFill('solid', start_color=HEADER_COLOR)
    ws['A1'].alignment = Alignment(horizontal='center', vertical='center')
    ws.row_dimensions[1].height = 30
    for ci, col in enumerate(df.columns, 1):
        c = ws.cell(row=2, column=ci)
        c.value = col
        c.font  = Font(name='Arial', bold=True, size=9, color='FFFFFF')
        c.fill  = PatternFill('solid', start_color='2E75B6')
        c.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        c.border = thin_border()
    ws.row_dimensions[2].height = 28
    for ri, row_data in enumerate(df.itertuples(index=False), 3):
        direction = str(row_data.Direction).upper()
        for ci, val in enumerate(row_data, 1):
            col_name   = df.columns[ci - 1]
            is_content = (col_name == 'Anonymized_Content')
            c = ws.cell(row=ri, column=ci)
            c.value  = val if val is not None else ''
            c.font   = Font(name='Arial', size=9)
            c.border = thin_border()
            c.alignment = Alignment(
                horizontal='left' if is_content else 'center',
                vertical='center', wrap_text=is_content
            )
            if col_name == 'Direction':
                c.fill = PatternFill('solid', start_color=IN_COLOR if direction=='IN' else OUT_COLOR)
                c.font = Font(name='Arial', size=9, bold=True)
            elif col_name == 'Transaction_type':
                c.fill = PatternFill('solid', start_color='EBF3FB')
            else:
                c.fill = PatternFill('solid', start_color=ALT_COLOR if ri%2==0 else 'FFFFFF')
        ws.row_dimensions[ri].height = 15
    for ci, col in enumerate(df.columns, 1):
        ws.column_dimensions[get_column_letter(ci)].width = COL_WIDTHS.get(col, 14)
    ws.freeze_panes = 'A3'
    wb.save(path)

print('Excel export helper ready!')


Excel export helper ready!


## 🚀 Step 10 — Run the full pipeline

In [10]:
owner_registry = {}
_user_ctr = 0

def get_owner_alias(name_norm):
    global _user_ctr
    if name_norm not in owner_registry:
        _user_ctr += 1
        owner_registry[name_norm] = (f'USER_{_user_ctr}', f'USER_{_user_ctr}_phone')
    return owner_registry[name_norm]

owner_map_records = []
summary_rows = []

print(f'Processing {len(files)} file(s)...')
print('-' * 70)

for filepath in files:
    print(f'\n{filepath.name}')

    try:
        df_raw = load_file(filepath)
    except Exception as e:
        print(f'  ERROR loading: {e}'); continue

    operator    = detect_operator(df_raw)
    owner_phone = detect_owner_phone(df_raw)
    owner_name  = detect_owner_name(df_raw, owner_phone) or owner_from_filename(filepath)
    owner_norm  = norm_name(owner_name) if owner_name else f'unknown_{filepath.stem}'
    user_id, user_id_phone = get_owner_alias(owner_norm)

    print(f'  Operator : {operator}')
    print(f'  Owner    : {owner_name or "(unknown)"} -> {user_id}')
    print(f'  Phone    : {owner_phone or "(unknown)"}')

    df_filtered = filter_balance_messages(df_raw)
    print(f'  Messages : {len(df_raw)} total -> {len(df_filtered)} balance-related')

    if len(df_filtered) == 0:
        print('  No balance-related messages found - skipping.'); continue

    records = []
    for _, row in df_filtered.iterrows():
        text = str(row['content'])
        tx_type, direction    = classify_transaction(text)
        amount, currency      = extract_amount(text)
        new_balance, bal_curr = extract_new_balance(text)
        if currency is None: currency = bal_curr
        anon = anonymize_message(text, owner_name, owner_phone, user_id)
        records.append({
            'UserId':             user_id,
            'Date':               row.get('date', ''),
            'Time':               row.get('time', ''),
            'Operator':           operator,
            'Transaction_type':   tx_type,
            'Direction':          direction,
            'Amount':             amount,
            'Currency':           currency,
            'New_balance':        new_balance,
            'Anonymized_Content': anon,
        })

    df_out = pd.DataFrame(records)
    n_in    = (df_out['Direction'] == 'IN').sum()
    n_out   = (df_out['Direction'] == 'OUT').sum()
    n_autre = (df_out['Transaction_type'] == 'autre').sum()
    print(f'  Extracted: {len(df_out)} tx | {n_in} IN | {n_out} OUT | {n_autre} unclassified')

    # Build output filename
    stem = filepath.stem
    sep_token = ' - ' if ' - ' in stem else ('_-_' if '_-_' in stem else None)
    if sep_token:
        prefix, _ = stem.rsplit(sep_token, 1)
        new_stem  = prefix + sep_token + user_id
    else:
        new_stem = f'{user_id}_{operator}'

    base_xlsx = OUTPUT_DIR / (new_stem + '.xlsx')
    base_csv  = OUTPUT_DIR / (new_stem + '.csv')
    ctr = 1
    while base_xlsx.exists() or base_csv.exists():
        base_xlsx = OUTPUT_DIR / (new_stem + f'_{ctr}.xlsx')
        base_csv  = OUTPUT_DIR / (new_stem + f'_{ctr}.csv')
        ctr += 1

    try:
        export_xlsx(df_out, base_xlsx, user_id, operator)
        df_out.to_csv(base_csv, index=False, encoding='utf-8-sig')
        print(f'  Saved -> {base_xlsx.name}')
        print(f'        -> {base_csv.name}')
    except Exception as e:
        print(f'  ERROR saving: {e}'); continue

    owner_map_records.append({
        'user_alias':    user_id,
        'phone_alias':   user_id_phone,
        'original_name': owner_name,
        'original_phone':owner_phone,
        'operator':      operator,
        'source_file':   filepath.name,
        'output_xlsx':   base_xlsx.name,
        'output_csv':    base_csv.name,
        'transactions':  len(df_out),
        'in_count':      int(n_in),
        'out_count':     int(n_out),
        'unclassified':  int(n_autre),
    })
    summary_rows.append({
        'File': filepath.name, 'Owner': owner_name or 'unknown',
        'User ID': user_id, 'Operator': operator,
        'Transactions': len(df_out), 'IN': int(n_in),
        'OUT': int(n_out), 'Unclassified': int(n_autre),
    })

with open(MAP_FILE, 'w', encoding='utf-8') as f:
    json.dump({'owner_map': owner_map_records}, f, ensure_ascii=False, indent=2)

print('\n' + '-' * 70)
print(f'Done! Outputs in: {OUTPUT_DIR}')
print(f'owner_map.json -> {len(owner_map_records)} record(s)')


Processing 23 file(s)...
----------------------------------------------------------------------

Messages avec MobileMoney 2026-03-20 21-32-36 - Madeleine Tchayo.csv
  Operator : MobileMoney
  Owner    : Madeleine Tchayo -> USER_1
  Phone    : 650528787
  Messages : 166 total -> 147 balance-related
  Extracted: 147 tx | 45 IN | 102 OUT | 0 unclassified
  Saved -> Messages avec MobileMoney 2026-03-20 21-32-36 - USER_1.xlsx
        -> Messages avec MobileMoney 2026-03-20 21-32-36 - USER_1.csv

Messages avec MobileMoney 2026-03-22 12_27_25 - Jacques Fah.csv
  Operator : MobileMoney
  Owner    : Jacques Fah -> USER_2
  Phone    : 650528787
  Messages : 638 total -> 315 balance-related
  Extracted: 315 tx | 79 IN | 236 OUT | 0 unclassified
  Saved -> Messages avec MobileMoney 2026-03-22 12_27_25 - USER_2.xlsx
        -> Messages avec MobileMoney 2026-03-22 12_27_25 - USER_2.csv

Messages avec OrangeMoney 2026-03-20 21-31-49 - Madeleine Tchayo.csv
  Operator : OrangeMoney
  Owner    : TCHAYO

## 📊 Step 11 — Run summary

In [11]:
if summary_rows:
    df_summary = pd.DataFrame(summary_rows)
    print('Processing Summary')
    print('-' * 70)
    display(df_summary)
else:
    print('No files were processed successfully.')


Processing Summary
----------------------------------------------------------------------


,File,Owner,User ID,Operator,Transactions,IN,OUT,Unclassified
0,Messages avec MobileMoney 2026-03-20 21-32-36 ...,Madeleine Tchayo,USER_1,MobileMoney,147,45,102,0
1,Messages avec MobileMoney 2026-03-22 12_27_25 ...,Jacques Fah,USER_2,MobileMoney,315,79,236,0
2,Messages avec OrangeMoney 2026-03-20 21-31-49 ...,TCHAYO,USER_3,OrangeMoney,894,203,682,9
3,Messages avec OrangeMoney 2026-03-20 22_05_33 ...,Bedibiki bougong,USER_4,OrangeMoney,147,9,133,5
4,Messages avec OrangeMoney 2026-03-21 11_16_17 ...,NDJANA MENGUE,USER_5,OrangeMoney,1525,119,1388,18
5,Messages avec OrangeMoney 2026-03-22 12_28_16 ...,FAH,USER_6,OrangeMoney,862,177,685,0
6,Messages avec OrangeMoney 2026-03-23 014528 - ...,ABOLO,USER_7,OrangeMoney,518,44,472,2
7,Messages avec OrangeMoney Brown 2026-03-23 083...,DJOUTSOP TAKOU,USER_8,OrangeMoney,58,9,46,3
8,Messages with MobileMoney 2026-03-17 214651.xlsx,unknown,USER_9,MobileMoney,63,25,38,0
9,Messages with MobileMoney 2026-03-20 21_13_48 ...,Dejon Fah Joël Xavier,USER_10,MobileMoney,455,58,397,0


## 🛠️ Step 12 — (Optional) Inspect unclassified messages

Run this after Step 10 to see messages that didn't match any rule.
Use the output to add new patterns to `TX_RULES` in Step 7.

In [12]:
# Change index to inspect a different file
target_file = files[0] if files else None

if target_file:
    _df = filter_balance_messages(load_file(target_file))
    unclassified = [
        str(row['content']) for _, row in _df.iterrows()
        if classify_transaction(str(row['content']))[0] == 'autre'
    ]
    if unclassified:
        print(f'{len(unclassified)} unclassified message(s) in {target_file.name}:\n')
        for i, msg in enumerate(unclassified[:10], 1):
            print(f'[{i}] {msg[:300]}\n')
    else:
        print('All messages classified!')


All messages classified!


## 🗂️ Step 13 — (Optional) View owner map

In [13]:
with open(MAP_FILE) as f:
    data = json.load(f)
df_map = pd.DataFrame(data['owner_map'])
print(f'owner_map.json - {len(df_map)} record(s)\n')
display(df_map[['user_alias','original_name','original_phone',
                'operator','transactions','in_count','out_count','unclassified']])


owner_map.json - 22 record(s)



,user_alias,original_name,original_phone,operator,transactions,in_count,out_count,unclassified
0,USER_1,Madeleine Tchayo,650528787,MobileMoney,147,45,102,0
1,USER_2,Jacques Fah,650528787,MobileMoney,315,79,236,0
2,USER_3,TCHAYO,699329183,OrangeMoney,894,203,682,9
3,USER_4,Bedibiki bougong,657604872,OrangeMoney,147,9,133,5
4,USER_5,NDJANA MENGUE,697692981,OrangeMoney,1525,119,1388,18
5,USER_6,FAH,699663071,OrangeMoney,862,177,685,0
6,USER_7,ABOLO,695247258,OrangeMoney,518,44,472,2
7,USER_8,DJOUTSOP TAKOU,656614565,OrangeMoney,58,9,46,3
8,USER_9,NaN,650528787,MobileMoney,63,25,38,0
9,USER_10,Dejon Fah JoÃ«l Xavier,650528787,MobileMoney,455,58,397,0
